# Preparing a Data Catalog for HydroMT-FIAT

This example shows how to create a data catalog YAML file for use with
HydroMT-FIAT. A data catalog describes where to find your data, how to
read it, and provides metadata for reproducibility.

We will cover catalog entries for all three main data types:
- **Hazard** (raster)
- **Exposure** (vector)
- **Vulnerability** (tabular)

In [ ]:
# Imports
from pathlib import Path

import yaml
from hydromt import log

from hydromt_fiat.data import fetch_data

log.initialize_logging()

## Inspect an existing data catalog

Let's first look at the example data catalog that ships with HydroMT-FIAT.

In [ ]:
# Fetch example data
global_dir = fetch_data("global-data")

# Read and display the catalog
catalog_path = Path(global_dir, "data_catalog.yml")
with open(catalog_path) as f:
    catalog_content = yaml.safe_load(f)

print(f"Catalog location: {catalog_path}")
print(f"\nNumber of entries: {len(catalog_content) - 1}")  # minus meta
print(f"\nEntries: {[k for k in catalog_content.keys() if k != 'meta']}")

## Structure of a data catalog

A data catalog YAML file has the following general structure:

```yaml
meta:
  roots:
    - ./path/to/data
  version: 1.0
  name: my_catalog

my_dataset_name:
  data_type: RasterDataset  # or GeoDataFrame, DataFrame
  uri: relative/path/to/file.tif
  driver:
    name: rasterio  # or pyogrio, pandas
  metadata:
    category: hazard
    crs: 4326
```

## Hazard data entry (Raster)

Hazard data is typically a raster file (GeoTIFF or NetCDF) containing
flood depth values on a spatial grid.

In [ ]:
# Example hazard catalog entry
hazard_entry = {
    "flood_event": {
        "data_type": "RasterDataset",
        "uri": "hazard/flood_map.tif",
        "driver": {
            "name": "rasterio",
            "filesystem": "local",
            "options": {
                "chunks": {"x": 1500, "y": 1500},
            },
        },
        "metadata": {
            "category": "hazard",
            "crs": 28992,
            "source_version": "1.0",
        },
    }
}

print("Hazard data catalog entry:")
print(yaml.dump(hazard_entry, default_flow_style=False))

## Exposure data entry (Vector)

Exposure data is a vector file (GeoPackage, FlatGeobuf, etc.) containing
building footprints or asset locations with attribute information.

In [ ]:
# Example exposure catalog entry
exposure_entry = {
    "buildings": {
        "data_type": "GeoDataFrame",
        "uri": "exposure/buildings.fgb",
        "driver": {
            "name": "pyogrio",
            "filesystem": "local",
        },
        "metadata": {
            "category": "exposure",
            "crs": 4326,
            "source_version": "1.0",
        },
    }
}

print("Exposure data catalog entry:")
print(yaml.dump(exposure_entry, default_flow_style=False))

## Vulnerability data entry (Tabular)

Vulnerability data consists of depth-damage curves stored as CSV tables.
A linking table maps object types to specific curves.

In [ ]:
# Example vulnerability catalog entries
vulnerability_entries = {
    "vulnerability_curves": {
        "data_type": "DataFrame",
        "uri": "vulnerability/curves.csv",
        "driver": {
            "name": "pandas",
            "filesystem": "local",
            "options": {
                "index_col": 0,
            },
        },
        "metadata": {
            "category": "vulnerability",
            "source_version": "1.0",
            "paper_doi": "10.2760/16510",
            "paper_ref": "Huizinga et al. (2017)",
        },
    },
    "vulnerability_curves_link": {
        "data_type": "DataFrame",
        "uri": "vulnerability/curves_link.csv",
        "driver": {
            "name": "pandas",
            "filesystem": "local",
            "options": {
                "index_col": 0,
            },
        },
        "metadata": {
            "category": "vulnerability",
        },
    },
}

print("Vulnerability data catalog entries:")
print(yaml.dump(vulnerability_entries, default_flow_style=False))

## Writing a complete catalog

Let's combine all entries into a complete catalog YAML file.

In [ ]:
# Combine into a full catalog
full_catalog = {
    "meta": {
        "roots": ["./data"],
        "version": "1.0",
        "name": "my_fiat_catalog",
    },
    **hazard_entry,
    **exposure_entry,
    **vulnerability_entries,
}

# Write to file
output_path = Path("my_data_catalog.yml")
with open(output_path, "w") as f:
    yaml.dump(full_catalog, f, default_flow_style=False, sort_keys=False)

print(f"Catalog written to: {output_path}")
print("\nContents:")
print(output_path.read_text())

## Using the catalog

Once you have a catalog, you can use it with both the CLI and Python API:

**CLI:**
```bash
hydromt build fiat ./model_output -i recipe.yml -d my_data_catalog.yml
```

**Python:**
```python
model = FIATModel(
    root="./model_output",
    mode="w+",
    data_libs=["my_data_catalog.yml"],
)
```

In [ ]:
# Verify the catalog loads correctly
# (This would work if the data files existed at the specified paths)
# dc = DataCatalog(data_libs=["my_data_catalog.yml"])
# print(dc)

# Clean up
output_path.unlink(missing_ok=True)
print("Done!")